In [7]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import normalize

In [ ]:
# Charge les fichiers .npy.
def load_npy_files(path_data="xvectors/voxblink2.npy", path_data_idx="xvectors/voxblink2_idx.npy"):
    data = np.load(path_data)
    data_idx = np.load(path_data_idx)

    return data, data_idx


data, data_idx = load_npy_files()

# Affiche les données et leurs dimensions.

print(data)
print(f"Shape : {data.shape}\n")

print(data_idx)
print(f"Shape : {data_idx.shape}")

[[ 4.7851820e-03 -3.3577152e-02  1.9579224e-02 ... -9.3641356e-03
   1.9330385e-03 -5.3961971e-04]
 [-6.7843804e-03 -2.2476003e-02  1.8925767e-02 ...  2.9063629e-04
  -2.7345007e-03  2.9082946e-03]
 [ 1.8351765e-03 -3.0638775e-02  1.8826926e-02 ... -2.6167207e-03
  -4.6694758e-03 -2.4883000e-03]
 ...
 [-1.4053062e-03 -1.5019094e-02 -1.0534138e-03 ...  2.3144640e-03
  -4.3242867e-03 -6.9390573e-03]
 [-9.9866251e-03 -9.1257365e-03  2.5506727e-03 ...  2.5071669e-03
  -3.7991968e-03 -2.4793537e-03]
 [-6.4502964e-03 -1.0802766e-02 -8.0625643e-05 ... -1.4390324e-03
  -6.3886005e-03 -4.9910615e-03]]
Shape : (259672, 96)

[   0    0    0 ... 2713 2713 2713]
Shape : (259672,)


In [ ]:
# Sépare les données en ensembles d'entraînement et de test.
X_train, X_test, y_train, y_test = train_test_split(data, data_idx, test_size=0.3, random_state=42)

In [ ]:
# Normalise l’espace original des x-vecteurs.
def x_vector_normalization(X_train, X_test):
    # Calcul de la moyenne sur l'ensemble d'entraînement.
    X_train_mean = np.mean(X_train, axis=0)
    
    # Soustrait la moyenne pour centrer les données.
    X_train_centered = X_train - X_train_mean
    X_test_centered = X_test - X_train_mean
    
    # Normalise les données centrées avec la norme L2.
    X_train_normalized = normalize(X_train_centered, norm="l2", axis=1)
    X_test_normalized = normalize(X_test_centered, norm="l2", axis=1)
    
    return X_train_normalized, X_test_normalized


X_train_norm, X_test_norm = x_vector_normalization(X_train, X_test)

In [18]:
# Implémente une baseline standard avec Orthogonal Matching Pursuit (OMP) sans rotation du résidu
# Pour pouvoir mesurer quantitativement le gain apporté par l’alignement dynamique.
import numpy as np
from sklearn.decomposition import DictionaryLearning
from sklearn.linear_model import orthogonal_mp


def omp_baseline(X_train, X_test, n_atoms=100, max_coefs=10):
    """Baseline OMP standard (sans rotation du résidu).

    Retourne la liste des MSE de reconstruction de k=1 à max_coefs.
    """
    # 1. Apprentissage du dictionnaire fixe D
    dict_learner = DictionaryLearning(
        n_components=n_atoms, transform_algorithm="omp", random_state=42
    )
    dictionary = dict_learner.fit(X_train).components_

    # 2. Calcul des MSE pour chaque nombre d'atomes k
    mses_omp = []
    for k in range(1, max_coefs + 1):
        H_test = orthogonal_mp(
            dictionary.T, X_test.T, n_nonzero_coefs=k
        ).T
        X_rec = H_test @ dictionary
        mse_k = np.mean((X_test - X_rec) ** 2)
        mses_omp.append(mse_k)

    return mses_omp


mses_omp = omp_baseline(X_train_norm, X_test_norm, n_atoms=100, max_coefs=10)
print("MSEs for OMP baseline (k=1 to 10):", mses_omp)


KeyboardInterrupt: 